In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import os
import sys
import json
import random

import numpy as np 
import pandas as pd 

import librosa as lb
import librosa.feature as lf
import librosa.display as ld
import soundfile as sf
import kagglehub 


import matplotlib.pyplot as plt
from IPython.display import Audio
from tqdm import tqdm


# Kaggle Set-up
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
#os.environ['HF_TOKEN'] = user_secrets.get_secret("hf_access")
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("kgg_user")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("kgg_key")

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
SR = 22050
DURATION = 30



# - Training & Validation melspectrogram creation

In [17]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
g = GENRES[9]


try:
    train_path = f"/kaggle/working/{g}/train"
    val_path = f"/kaggle/working/{g}/val"
    os.makedirs(train_path, exist_ok=True)
    os.makedirs(val_path, exist_ok=True)
    
    print(f"Directory created at: {train_path}  and {val_path}")
except Exception as e:
    print(f"Error creating directory: {e}")


root_dir_path_train = f"/kaggle/input/datasets/akashkumbhakar/{g}-augmented-5000-mashup-train"
root_dir_path_val = f"/kaggle/input/datasets/akashkumbhakar/{g}-augmented-500-mashup-val"

print("Train set for : ", g)
# train mel-spectrogram
for i in tqdm(range(0,5000),desc=f"Creating Training mel-spec of {g}..."):
    file_name = f"mashup_v2_{i}.wav"
    path = os.path.join(root_dir_path_train,file_name)
    y,sr = lb.load(path)
    mel = lf.melspectrogram(y=y,sr=SR,n_fft=1024,hop_length=1024,n_mels=128)
    log_mel_spec = lb.power_to_db(mel, ref=np.max)
    np.save(f"{train_path}/mashup_v2_{i}.npy", log_mel_spec)
    

print("Validation set for : ", g)
# val mel-spectrogram
for i in tqdm(range(0,500),desc=f"Creating validation mel-spec of {g}..."):
    file_name = f"mashup_v2_{i}.wav"
    path = os.path.join(root_dir_path_val,file_name)
    y,sr = lb.load(path)
    mel = lf.melspectrogram(y=y,sr=SR,n_fft=1024,hop_length=1024,n_mels=128)
    log_mel_spec = lb.power_to_db(mel, ref=np.max)
    np.save(f"{val_path}/mashup_v2_{i}.npy", log_mel_spec)

print("✅ complete.")

Directory created at: /kaggle/working/rock/train  and /kaggle/working/rock/val
Train set for :  rock


Creating Training mel-spec of rock...: 100%|██████████| 5000/5000 [04:16<00:00, 19.46it/s]


Validation set for :  rock


Creating validation mel-spec of rock...: 100%|██████████| 500/500 [00:24<00:00, 20.33it/s]

✅ complete.


In [18]:
mel = np.load(f"{train_path}/mashup_v2_1.npy")
print(type(mel),mel.shape,mel.dtype)
mel = np.load(f"{val_path}/mashup_v2_1.npy")
print(type(mel),mel.shape,mel.dtype)

<class 'numpy.ndarray'> (128, 646) float32
<class 'numpy.ndarray'> (128, 646) float32


In [19]:
# Storing to kaggle hub
handle = f'akashkumbhakar/{g}-mel-T5000-V500'
local_dataset= f'/kaggle/working/{g}'

# Create a new dataset
kagglehub.dataset_upload(
    handle, 
    local_dataset,
    version_notes="5000 train and 500 validation mel-spectrogram from newly augmented musics"
)

Uploading Dataset https://api.kaggle.com/datasets/akashkumbhakar/rock-mel-T5000-V500 ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpflsjr1bg/archive.zip


Uploading: 100%|██████████| 1.82G/1.82G [00:14<00:00, 126MB/s]

Upload successful: /tmp/tmpflsjr1bg/archive.zip (2GB)


Your dataset has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/akashkumbhakar/rock-mel-T5000-V500


In [21]:
import shutil
local_dataset = f'/kaggle/working/{g}'
if os.path.exists(local_dataset):
    shutil.rmtree(local_dataset)
    print(f"{local_dataset} Removed")
else:
    print(f"PATH : {local_dataset} not exist OR alreaady removed.")

/kaggle/working/pop Removed
